# Extension: does the finding generalise? (2nd model + DoRA)

The original study found that on **Qwen2.5-1.5B / Alpaca**, LoRA makes *tiny (≈0.4–1.5%), high-rank, attention-biased edits that rescale rather than rotate representations* — but it was explicitly exploratory: one model, one PEFT variant.

This notebook turns that anecdote into a claim (or refutes it) along two axes:

1. **A second model family** — [SmolLM2-1.7B](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B) (ungated, Llama-style architecture).
2. **A second PEFT variant** — **DoRA** (Liu et al., 2024), which decomposes the update into magnitude and direction.

It also adds the **behavioural validation** the original study lacked: held-out instruction loss (base vs fine-tuned).

**Runtime**: 4 configs (2 models × LoRA/DoRA). ~40 min on an **A100/L4**, 2–3 h on a free T4. Reduce `MAX_SAMPLES` for a faster smoke run.

> ⚠️ **Cómo ejecutarlo**: celdas en orden, de arriba abajo. Todas las celdas son *idempotentes* (puedes re-ejecutar cualquiera sin romper nada). Si algo se queda en mal estado: *Entorno de ejecución → Reiniciar* y vuelve a ejecutar desde arriba.

In [1]:
# Setup robusto e idempotente: siempre parte de /content, clona limpio y purga
# cualquier versión vieja del paquete que el kernel tenga cacheada.
import os, shutil, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO = "/content/Interpreting-LoRA-Fine-Tuning"

os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q https://github.com/mlahozy21/Interpreting-LoRA-Fine-Tuning.git
os.chdir(REPO)

!pip install -q -U "transformers>=4.44" "peft>=0.13" "datasets>=2.18" accelerate
!pip uninstall -y -q torchao   # el torchao preinstalado en Colab es incompatible con peft

# purgar módulos viejos y dejar solo la ruta absoluta del clon nuevo
for k in list(sys.modules):
    if k.startswith("lora_interp"):
        del sys.modules[k]
sys.path[:] = [p for p in sys.path if "Interpreting-LoRA" not in str(p)]
sys.path.insert(0, f"{REPO}/src")

# verificación: si esto falla, el código clonado no es el esperado
import inspect
from lora_interp.train import train_lora
assert "use_dora" in inspect.signature(train_lora).parameters, \
    "train.py sin use_dora: haz push del repo actualizado antes de continuar"
print("setup OK — cwd:", os.getcwd())
print("train_lora:", inspect.signature(train_lora))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.2 MB/s eta 0:00:00
setup OK — cwd: /content/Interpreting-LoRA-Fine-Tuning
train_lora: (model_name='Qwen/Qwen2.5-1.5B', rank=16, target='all', dataset='tatsu-lab/alpaca', max_samples=2000, epochs=1.0, lr=0.0002, batch_size=8, grad_accum=2, max_len=512, output_dir=None, seed=42, use_dora=False, grad_checkpoint=False)


In [2]:
import copy, gc, json

import numpy as np
import torch

from lora_interp.analysis import exact_update_norms, representation_drift
from lora_interp.train import train_lora, _format
from lora_interp.utils import set_seed, load_base

MODELS = ["Qwen/Qwen2.5-1.5B", "HuggingFaceTB/SmolLM2-1.7B"]
VARIANTS = [("lora", False), ("dora", True)]
RANK = 16
TARGET = "all"
MAX_SAMPLES = 2000          # reduce a 500 para una prueba rápida
SEED = 42

HELDOUT_DRIFT = [
    "### Instruction:\nExplain what overfitting is.\n\n### Response:\n",
    "### Instruction:\nSummarise the causes of the French Revolution.\n\n### Response:\n",
    "### Instruction:\nWrite a Python function that reverses a string.\n\n### Response:\n",
    "### Instruction:\nGive three tips for sleeping better.\n\n### Response:\n",
    "### Instruction:\nWhat is the capital of Australia?\n\n### Response:\n",
]

# batch segun la GPU: A100 (8,2) · L4 (4,4) · T4 (2,8) — mismo batch efectivo 16
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
BS, GA = (8, 2) if gpu_gb > 30 else (4, 4) if gpu_gb > 20 else (2, 8)
GC = gpu_gb < 30   # gradient checkpointing en T4/L4 (menos memoria, algo mas lento)
print(f"GPU: {gpu_gb:.0f} GB -> batch_size={BS}, grad_accum={GA}, grad_checkpoint={GC}")

print("device:", "cuda" if torch.cuda.is_available() else "cpu")

GPU: 24 GB -> batch_size=4, grad_accum=4, grad_checkpoint=True
device: cuda


## Behavioural validation: held-out instruction loss

Mean token-level cross-entropy on 200 *held-out* Alpaca examples (never used in training — training takes the first `MAX_SAMPLES` of the seeded shuffle, we take the next 200).

In [3]:
from datasets import load_dataset

_alpaca = load_dataset("tatsu-lab/alpaca", split="train").shuffle(seed=SEED)
EVAL_EXAMPLES = [_format(e) for e in _alpaca.select(range(MAX_SAMPLES, MAX_SAMPLES + 200))]


@torch.no_grad()
def heldout_loss(model, tok, texts, max_len=512):
    model.eval()
    losses = []
    for t in texts:
        ids = tok(t + tok.eos_token, return_tensors="pt", truncation=True,
                  max_length=max_len).input_ids.to(model.device)
        out = model(ids, labels=ids)
        losses.append(out.loss.item())
    return float(np.mean(losses))

print(f"{len(EVAL_EXAMPLES)} ejemplos held-out listos")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

200 ejemplos held-out listos


## Exact update norms — desde los factores del adaptador, en fp32

`exact_update_norms` calcula la actualización **efectiva exacta** `dW = W_eff - W0` por módulo, en fp32, a partir de los tensores del adaptador. Para LoRA esto coincide con la actualización dirigida `dV = (alpha/r)·B@A`; para **DoRA incluye además el reescalado por el vector de magnitud** `m`, de modo que es el diagnóstico que **sí distingue DoRA de LoRA**. Se reporta su norma relativa y su *participation ratio* (alias `effective_rank`).

> ⚠️ **No usamos `merge_and_unload()` para medir el rango.** Fusionar guarda el peso en bf16; restar ese peso fusionado del base recupera un `dW` contaminado por el redondeo de bf16 en todas las dimensiones, lo que **infla el rango efectivo de ~16 a varios cientos** (artefacto numérico). En su lugar reconstruimos `dW` analíticamente en fp32 desde los factores LoRA (`A`, `B`, `scaling`) y, si está presente, el vector de magnitud de DoRA — sin pérdida de precisión y sin contaminar el rango.


In [4]:
# (El rango efectivo y la magnitud se calculan con exact_update_norms
#  de src/lora_interp/analysis.py: dW = W_eff - W0 en fp32 desde los
#  factores del adaptador, incluyendo la magnitud de DoRA cuando existe.
#  Se evita merge_and_unload para no contaminar el rango con bf16.)


## Run the grid (2 models × LoRA/DoRA)

Idempotente: guarda cada configuración terminada en `results_partial.json`; si la celda se interrumpe y la relanzas, salta las configuraciones ya hechas.

In [5]:
import os

ATTN_TYPES = {"q_proj", "k_proj", "v_proj", "o_proj"}
PARTIAL = "results_partial.json"
os.makedirs("figures", exist_ok=True)

results = json.load(open(PARTIAL)) if os.path.exists(PARTIAL) else []
done = {(r["model"], r["variant"]) for r in results}
base_losses = {}

for model_name in MODELS:
    short = model_name.split("/")[-1]
    for variant, use_dora in VARIANTS:
        if (short, variant) in done:
            print(f"{short}/{variant}: ya completado, salto")
            continue
        tag = f"{short}_{variant}_r{RANK}_{TARGET}"
        print(f"\n{'='*70}\n{tag}\n{'='*70}")
        model, tok = train_lora(model_name=model_name, rank=RANK, target=TARGET,
                                max_samples=MAX_SAMPLES, epochs=1.0,
                                output_dir=f"outputs/{tag}", seed=SEED,
                                use_dora=use_dora,
                                batch_size=BS, grad_accum=GA, grad_checkpoint=GC)

        # 1) loss base: mismo modelo con el adaptador desactivado (cero copias extra)
        if short not in base_losses:
            with model.disable_adapter():
                base_losses[short] = heldout_loss(model, tok, EVAL_EXAMPLES)
        # 2) validación conductual del fine-tuned
        ft_loss = heldout_loss(model, tok, EVAL_EXAMPLES)
        # 3) drift (adapter on/off) — antes del merge
        drift = representation_drift(model, tok, HELDOUT_DRIFT)
        # 4) magnitud + rango efectivo de la actualizacion EXACTA dW = W_eff - W0,
        #    en fp32 desde los factores del adaptador. Para LoRA coincide con
        #    dV=(alpha/r)*B@A; para DoRA INCLUYE el reescalado por el vector de
        #    magnitud, asi que distingue DoRA de LoRA. No se hace merge en bf16
        #    (que inflaria el rango por ruido de redondeo); todo se calcula en fp32.
        rows = exact_update_norms(model)
        json.dump(rows, open(f"figures/update_norms_{tag}.json", "w"), indent=1)

        attn_u = [r["rel_update"] for r in rows if r["type"] in ATTN_TYPES]
        mlp_u = [r["rel_update"] for r in rows if r["type"] not in ATTN_TYPES]
        results.append({
            "model": short, "variant": variant,
            "rank": RANK, "target": TARGET,
            "heldout_loss_base": round(base_losses[short], 4),
            "heldout_loss_ft": round(ft_loss, 4),
            "mean_rel_update": round(float(np.mean([r["rel_update"] for r in rows])), 4),
            "rel_update_attn": round(float(np.mean(attn_u)), 4),
            "rel_update_mlp": round(float(np.mean(mlp_u)), 4),
            "mean_eff_rank": round(float(np.mean([r["effective_rank"] for r in rows])), 2),
            "drift_cos_final": round(drift["cosine_similarity"][-1], 4),
            "drift_relL2_final": round(drift["relative_l2"][-1], 4),
        })
        json.dump(results, open(PARTIAL, "w"), indent=1)   # checkpoint
        print(results[-1])

        # limpieza agresiva entre configuraciones
        del model, rows
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print(f"GPU libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

print(f"\n{len(results)}/4 configuraciones completadas")



Qwen2.5-1.5B_lora_r16_all


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

trainable params: 18,726,912 || all params: 1,730,103,296 || trainable%: 1.0824


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
25,1.524392
50,1.340915
75,1.350324
100,1.321475
125,1.306435


{'model': 'SmolLM2-1.7B', 'variant': 'dora', 'rank': 16, 'target': 'all', 'heldout_loss_base': 2.2294, 'heldout_loss_ft': 1.3709, 'mean_rel_update': 0.0022, 'rel_update_attn': 0.0023, 'rel_update_mlp': 0.002, 'mean_eff_rank': 10.21, 'drift_cos_final': 0.9282, 'drift_relL2_final': 0.3729}
GPU libre: 23.3 GB

4/4 configuraciones completadas


## Results

In [6]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("results_extension.csv", index=False)
print(df.to_markdown(index=False))

| model        | variant   |   rank | target   |   heldout_loss_base |   heldout_loss_ft |   mean_rel_update |   rel_update_attn |   rel_update_mlp |   mean_eff_rank |   drift_cos_final |   drift_relL2_final |
|:-------------|:----------|-------:|:---------|--------------------:|------------------:|------------------:|------------------:|-----------------:|----------------:|------------------:|--------------------:|
| Qwen2.5-1.5B | lora      |     16 | all      |              1.9758 |            1.4542 |            0.0063 |            0.0068 |           0.0056 |           13.4  |            0.9714 |              0.2563 |
| Qwen2.5-1.5B | dora      |     16 | all      |              1.9758 |            1.454  |            0.0062 |            0.0068 |           0.0056 |           13.51 |            0.9711 |              0.2592 |
| SmolLM2-1.7B | lora      |     16 | all      |              2.2294 |            1.3742 |            0.0022 |            0.0024 |           0.002  |           

## What to check against the original four findings

1. **Tiny updates** — ¿`mean_rel_update` sigue ≲ 1–2% en SmolLM2?
2. **Full rank budget** — ¿`mean_eff_rank` ≈ r (16)? (medido desde BA en fp32; no desde el peso fusionado en bf16).
3. **Attention bias** — ¿`rel_update_attn > rel_update_mlp` en ambos modelos y variantes?
4. **Rescale, not rotate** — ¿`drift_cos_final` alto (≥0.92) con `drift_relL2_final` apreciable?

Más el chequeo conductual: `heldout_loss_ft < heldout_loss_base` en cada fila.

**Next step**: pega `results_extension.csv` como segunda tabla en el README bajo "Results" y actualiza el párrafo de *Limitations*.

> 💾 Antes de cerrar Colab, descarga `results_extension.csv`.